### 코트라 메인페이지에서 특정 검색어 검색 후 들어가는 것을 의도

In [ ]:
# pip install --upgrade numpy pandas --force-reinstall

  Using cached numpy-2.4.3-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-2.4.3-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
Using cached pandas-3.0.1-cp311-cp311-macosx_11_0_arm64.whl (9.9 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.0
    Uninstalling numpy-2.4.0:
      Successfully uninstalled numpy-2.4.0
  Attempting uninstall: python-dateutil;5;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
    Found existing installation: python-dateutil 2.9.0

In [ ]:
# pip install lxml tabulate

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# pip install pandas openpyxl tabulate lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import time
import requests
import pandas as pd
import io
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options

# 1. 환경 설정
PROJECT_NAME = "KOTRA_Proper_Table_Project"
IMAGE_DIR = os.path.join(PROJECT_NAME, "images")
os.makedirs(IMAGE_DIR, exist_ok=True)

options = Options()
driver = webdriver.Chrome(options=options)
BASE_URL = "https://dream.kotra.or.kr"

def flatten_cols(df):
    """지저분한 2층 제목(MultiIndex)을 한 줄로 합쳐주는 함수"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            '_'.join([str(c) for c in col if 'Unnamed' not in str(c)]).strip()
            for col in df.columns.values
        ]
    return df

try:
    print("🚀 KOTRA 검색 가동: '일본 기초화장품'...")
    driver.get(f"{BASE_URL}/kotranews/index.do")
    time.sleep(3)

    search_box = driver.find_element(By.CLASS_NAME, "search_inp")
    search_box.send_keys("일본 기초화장품")
    search_box.send_keys(Keys.RETURN)
    time.sleep(3)

    # 상위 2개 기사 선정
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    result_boxes = soup.select('div.result_box')[:2]
    article_links = [BASE_URL + b.find('a')['href'] for b in result_boxes if b.find('a')]

    print(f"✅ 대상 기사 {len(article_links)}개 분석 시작\n")

    must_keywords = ["일본", "기초화장품"]

    for idx, url in enumerate(article_links, 1):
        print(f"--- [{idx}번 기사: 데이터 정규화 작업] ---")
        driver.get(url)
        time.sleep(3)
        
        full_html = driver.page_source
        detail_soup = BeautifulSoup(full_html, 'html.parser')
        contents = detail_soup.select_one('div#contents')
        if not contents: continue

        # [필터링]
        all_text = contents.get_text(strip=True).replace(" ", "")
        if not all(word in all_text for word in must_keywords):
            print("🔴 키워드 조건 미달 스킵")
            continue

        # [개선사항 1] 표를 엑셀로 개별 저장 (진짜 '정규표' 생성)
        tables = contents.find_all('table')
        table_md_list = [] # 텍스트 파일 기록용

        if tables:
            for t_idx, table in enumerate(tables):
                try:
                    df = pd.read_html(io.StringIO(str(table)))[0]
                    # [개선사항 2] 지저분한 멀티 헤더 정리
                    df = flatten_cols(df)
                    
                    # 엑셀 파일로 물리적 저장 (가장 깔끔한 결과물)
                    excel_path = os.path.join(PROJECT_NAME, f"article_{idx}_table_{t_idx+1}.xlsx")
                    df.to_excel(excel_path, index=False)
                    
                    # 텍스트 파일용 요약 (마크다운)
                    table_md_list.append(f"\n[표 {t_idx+1} 정보]\n{df.to_markdown(index=False)}\n")
                    
                    # 중복 방지를 위해 본문에서 표 태그 삭제
                    table.decompose()
                except: pass
            print(f"  📊 {len(tables)}개의 표를 엑셀 파일로 정밀 추출 완료")

        # [개선사항 3] 이미지 및 텍스트 수집
        img_tags = contents.find_all('img')
        img_saved = 0
        for i_idx, img in enumerate(img_tags):
            src = img.get('src')
            if src:
                if src.startswith('/'): src = BASE_URL + src
                try:
                    res = requests.get(src, timeout=5)
                    with open(os.path.join(IMAGE_DIR, f"art_{idx}_img_{i_idx}.jpg"), 'wb') as f:
                        f.write(res.content)
                    img_saved += 1
                except: pass

        # 최종 요약 리포트 저장
        report_path = os.path.join(PROJECT_NAME, f"article_{idx}_summary.txt")
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(f"URL: {url}\n")
            f.write("\n### [STRUCTURED TABLES]\n")
            f.write("".join(table_md_list) if table_md_list else "표 없음")
            f.write("\n\n### [CLEANED TEXT]\n")
            f.write(contents.get_text(separator='\n', strip=True))

        print(f"  💾 저장 완료: 엑셀 {len(tables)}개, 이미지 {img_saved}개, 요약문 1개\n")

except Exception as e:
    print(f"❌ 에러 발생: {e}")
finally:
    driver.quit()
    print("🏁 데이터 정규화 및 크롤링 파이프라인 종료")

🚀 KOTRA 검색 가동: '일본 기초화장품'...
✅ 대상 기사 1개 분석 시작

--- [1번 기사: 데이터 정규화 작업] ---
  📊 5개의 표를 엑셀 파일로 정밀 추출 완료
  💾 저장 완료: 엑셀 5개, 이미지 3개, 요약문 1개

🏁 데이터 정규화 및 크롤링 파이프라인 종료


In [2]:
import os
import io
import time
import sqlite3
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 1. 기본 설정
# =========================
PROJECT_NAME = "KOTRA_Proper_Table_Project"
IMAGE_DIR = os.path.join(PROJECT_NAME, "images")
TABLE_DIR = os.path.join(PROJECT_NAME, "tables")
REPORT_DIR = os.path.join(PROJECT_NAME, "reports")
DB_PATH = os.path.join(PROJECT_NAME, "kotra_articles.db")

os.makedirs(PROJECT_NAME, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

BASE_URL = "https://dream.kotra.or.kr"
SEARCH_URL = f"{BASE_URL}/kotranews/index.do"

SEARCH_KEYWORD = "일본 기초화장품"
MUST_KEYWORDS = ["일본", "기초화장품"]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

# 크롬 옵션
options = Options()
# options.add_argument("--headless=new")  # 필요하면 주석 해제
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)


# =========================
# 2. 유틸 함수
# =========================
def flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    """멀티헤더(MultiIndex) 컬럼을 한 줄로 정리"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)]).strip("_")
            for col in df.columns.values
        ]
    else:
        df.columns = [str(c).strip() for c in df.columns]
    return df


def get_image_extension(response: requests.Response, img_url: str) -> str:
    """이미지 확장자 자동 판별"""
    content_type = response.headers.get("Content-Type", "").lower()

    if "png" in content_type:
        return "png"
    elif "jpeg" in content_type or "jpg" in content_type:
        return "jpg"
    elif "webp" in content_type:
        return "webp"
    elif "gif" in content_type:
        return "gif"

    # content-type이 애매하면 URL 기준 보조 판별
    path = urlparse(img_url).path.lower()
    if path.endswith(".png"):
        return "png"
    elif path.endswith(".jpg") or path.endswith(".jpeg"):
        return "jpg"
    elif path.endswith(".webp"):
        return "webp"
    elif path.endswith(".gif"):
        return "gif"

    return "jpg"


def create_db(conn: sqlite3.Connection):
    """DB 테이블 생성"""
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS articles (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        keyword TEXT,
        article_order INTEGER,
        title TEXT,
        url TEXT UNIQUE,
        raw_text TEXT,
        cleaned_text TEXT,
        table_count INTEGER,
        image_count INTEGER,
        collected_at TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS article_tables (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        article_url TEXT,
        table_order INTEGER,
        excel_path TEXT,
        markdown_text TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS article_images (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        article_url TEXT,
        image_order INTEGER,
        image_url TEXT,
        saved_path TEXT
    )
    """)

    conn.commit()


def insert_article(conn, article_data: dict):
    cur = conn.cursor()
    cur.execute("""
    INSERT OR REPLACE INTO articles
    (keyword, article_order, title, url, raw_text, cleaned_text, table_count, image_count, collected_at)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, datetime('now'))
    """, (
        article_data["keyword"],
        article_data["article_order"],
        article_data["title"],
        article_data["url"],
        article_data["raw_text"],
        article_data["cleaned_text"],
        article_data["table_count"],
        article_data["image_count"]
    ))
    conn.commit()


def insert_table_info(conn, table_data: dict):
    cur = conn.cursor()
    cur.execute("""
    INSERT INTO article_tables
    (article_url, table_order, excel_path, markdown_text)
    VALUES (?, ?, ?, ?)
    """, (
        table_data["article_url"],
        table_data["table_order"],
        table_data["excel_path"],
        table_data["markdown_text"]
    ))
    conn.commit()


def insert_image_info(conn, image_data: dict):
    cur = conn.cursor()
    cur.execute("""
    INSERT INTO article_images
    (article_url, image_order, image_url, saved_path)
    VALUES (?, ?, ?, ?)
    """, (
        image_data["article_url"],
        image_data["image_order"],
        image_data["image_url"],
        image_data["saved_path"]
    ))
    conn.commit()


# =========================
# 3. 검색 결과 수집
# =========================
def search_kotra_articles(keyword: str, top_n: int = 2):
    print(f"🚀 KOTRA 검색 시작: '{keyword}'")

    driver.get(SEARCH_URL)

    search_box = wait.until(
        EC.presence_of_element_located((By.CLASS_NAME, "search_inp"))
    )
    search_box.clear()
    search_box.send_keys(keyword)
    search_box.send_keys(Keys.RETURN)

    wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.result_box"))
    )

    soup = BeautifulSoup(driver.page_source, "html.parser")
    result_boxes = soup.select("div.result_box")[:top_n]

    article_links = []
    for box in result_boxes:
        a_tag = box.find("a")
        if a_tag and a_tag.get("href"):
            full_url = urljoin(BASE_URL, a_tag["href"])
            article_links.append(full_url)

    print(f"✅ 대상 기사 {len(article_links)}개 수집 완료\n")
    return article_links


# =========================
# 4. 기사 상세 파싱
# =========================
def parse_article(url: str, article_order: int, keyword: str, must_keywords: list, conn):
    print(f"--- [{article_order}번 기사] 분석 시작 ---")
    print(f"URL: {url}")

    driver.get(url)

    wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div#contents"))
    )

    detail_soup = BeautifulSoup(driver.page_source, "html.parser")

    contents = detail_soup.select_one("div#contents")
    if not contents:
        print("❌ 본문 영역(div#contents)을 찾지 못했어.\n")
        return None

    # 제목 추출
    title_tag = detail_soup.select_one("h1")
    title = title_tag.get_text(strip=True) if title_tag else "제목 없음"

    # 원문 텍스트
    raw_text = contents.get_text(separator="\n", strip=True)
    compact_text = raw_text.replace(" ", "").replace("\n", "")

    # 키워드 필터링
    if not all(word in compact_text for word in must_keywords):
        print(f"🔴 키워드 조건 미달로 스킵: {must_keywords}\n")
        return None

    # -------------------------
    # 표 추출
    # -------------------------
    tables = contents.find_all("table")
    table_md_list = []
    saved_table_count = 0

    for t_idx, table in enumerate(tables, start=1):
        try:
            df = pd.read_html(io.StringIO(str(table)))[0]
            df = flatten_cols(df)

            excel_path = os.path.join(
                TABLE_DIR,
                f"article_{article_order}_table_{t_idx}.xlsx"
            )
            df.to_excel(excel_path, index=False)

            table_md = df.to_markdown(index=False)
            table_md_list.append(f"\n[표 {t_idx}]\n{table_md}\n")

            insert_table_info(conn, {
                "article_url": url,
                "table_order": t_idx,
                "excel_path": excel_path,
                "markdown_text": table_md
            })

            saved_table_count += 1

        except Exception as e:
            print(f"⚠️ 표 {t_idx} 처리 실패: {e}")

    # 표 제거 후 본문 정리
    for table in contents.find_all("table"):
        table.decompose()

    # -------------------------
    # 이미지 추출
    # -------------------------
    img_tags = contents.find_all("img")
    saved_image_count = 0

    for i_idx, img in enumerate(img_tags, start=1):
        src = img.get("src")
        if not src:
            continue

        img_url = urljoin(BASE_URL, src)

        try:
            res = requests.get(img_url, headers=HEADERS, timeout=10)
            res.raise_for_status()

            ext = get_image_extension(res, img_url)
            img_path = os.path.join(
                IMAGE_DIR,
                f"article_{article_order}_img_{i_idx}.{ext}"
            )

            with open(img_path, "wb") as f:
                f.write(res.content)

            insert_image_info(conn, {
                "article_url": url,
                "image_order": i_idx,
                "image_url": img_url,
                "saved_path": img_path
            })

            saved_image_count += 1

        except Exception as e:
            print(f"⚠️ 이미지 {i_idx} 저장 실패: {img_url} / {e}")

    # -------------------------
    # 정리된 본문
    # -------------------------
    cleaned_text = contents.get_text(separator="\n", strip=True)

    # -------------------------
    # 리포트 저장
    # -------------------------
    report_path = os.path.join(REPORT_DIR, f"article_{article_order}_summary.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"KEYWORD: {keyword}\n")
        f.write(f"TITLE: {title}\n")
        f.write(f"URL: {url}\n")
        f.write(f"TABLE_COUNT: {saved_table_count}\n")
        f.write(f"IMAGE_COUNT: {saved_image_count}\n")

        f.write("\n\n### [STRUCTURED TABLES]\n")
        f.write("".join(table_md_list) if table_md_list else "표 없음")

        f.write("\n\n### [CLEANED TEXT]\n")
        f.write(cleaned_text)

    # -------------------------
    # DB 저장
    # -------------------------
    article_data = {
        "keyword": keyword,
        "article_order": article_order,
        "title": title,
        "url": url,
        "raw_text": raw_text,
        "cleaned_text": cleaned_text,
        "table_count": saved_table_count,
        "image_count": saved_image_count
    }
    insert_article(conn, article_data)

    print(f"✅ 제목: {title}")
    print(f"📊 저장된 표: {saved_table_count}개")
    print(f"🖼 저장된 이미지: {saved_image_count}개")
    print(f"💾 리포트 저장: {report_path}\n")

    return article_data


# =========================
# 5. 메인 실행
# =========================
conn = sqlite3.connect(DB_PATH)
create_db(conn)

all_articles = []

try:
    article_links = search_kotra_articles(SEARCH_KEYWORD, top_n=2)

    for idx, article_url in enumerate(article_links, start=1):
        result = parse_article(
            url=article_url,
            article_order=idx,
            keyword=SEARCH_KEYWORD,
            must_keywords=MUST_KEYWORDS,
            conn=conn
        )
        if result:
            all_articles.append(result)

except Exception as e:
    print(f"❌ 전체 파이프라인 에러: {e}")

finally:
    driver.quit()
    conn.close()
    print("🏁 크롤링 및 DB 저장 파이프라인 종료")


# =========================
# 6. 결과 확인용 DataFrame
# =========================
df_articles = pd.DataFrame(all_articles)
df_articles

🚀 KOTRA 검색 시작: '일본 기초화장품'
✅ 대상 기사 1개 수집 완료

--- [1번 기사] 분석 시작 ---
URL: https://dream.kotra.or.kr/kotranews/cms/news/actionKotraBoardDetail.do?SITE_NO=3&MENU_ID=190&CONTENTS_NO=2&bbsGbn=254&bbsSn=254&pNttSn=232461
✅ 제목: 
📊 저장된 표: 5개
🖼 저장된 이미지: 3개
💾 리포트 저장: KOTRA_Proper_Table_Project/reports/article_1_summary.txt

🏁 크롤링 및 DB 저장 파이프라인 종료


,keyword,article_order,title,url,raw_text,cleaned_text,table_count,image_count
0,일본 기초화장품,1,,https://dream.kotra.or.kr/kotranews/cms/news/a...,홈\n상품·산업\n뉴스\n상품·산업\n국가·지역정보\n보고서\n멀티미디어뉴스\n해외...,홈\n상품·산업\n뉴스\n상품·산업\n국가·지역정보\n보고서\n멀티미디어뉴스\n해외...,5,3
